In [1]:
import hashlib
import os
import platform
import uuid
import sys
# 修改为自己的学号
STUDENT_ID = 20244081101
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]\t{STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")
print(f"[Python] {sys.version.split()[0]}")

[Conda 环境] pyexp
[学号]	20244081101
[机器指纹] d24d45b16bc06106
[RUN_ID] 20244081101-d24d45b16bc06106
[Python] 3.14.6


## 实验1：正则表达式的基础匹配

下面在三段文本上分别使用数字、空白、单词边界、选择和量词。

In [2]:
import re

text_basic = "cat on mat; later: cats and dogs"
print("原文:", text_basic)
print(r"\d+ ->", re.findall(r"\d+", text_basic))
print(r"\s+ ->", re.findall(r"\s+", text_basic))
print(r"\bcat\b ->", re.findall(r"\bcat\b", text_basic))

text_alt = "cat dog bird CAT"
print("原文:", text_alt)
print(r"cat|dog ->", re.findall(r"cat|dog", text_alt))
print(r"a|aa|aaa ->", re.findall(r"a|aa|aaa", "aaaa"))
print(r"aaa|aa|a ->", re.findall(r"aaa|aa|a", "aaaa"))

text_quant = "ct cat caat caaat"
print("原文:", text_quant)
for pattern in [r'ca*t', r'ca+t', r'ca?t', r'ca{2}t', r'ca{1,2}t']:
    print(f"{pattern} -> {re.findall(pattern, text_quant)}")

原文: cat on mat; later: cats and dogs
\d+ -> []
\s+ -> [' ', ' ', ' ', ' ', ' ', ' ']
\bcat\b -> ['cat']
原文: cat dog bird CAT
cat|dog -> ['cat', 'dog']
a|aa|aaa -> ['a', 'a', 'a', 'a']
aaa|aa|a -> ['aaa', 'a']
原文: ct cat caat caaat
ca*t -> ['ct', 'cat', 'caat', 'caaat']
ca+t -> ['cat', 'caat', 'caaat']
ca?t -> ['ct', 'cat']
ca{2}t -> ['caat']
ca{1,2}t -> ['cat', 'caat']


这段文本里没有数字，所以 `\d+` 的结果是空列表。`\s+` 匹配到单词之间的空格。`\bcat\b` 只匹配独立的单词 cat，不会匹配后面的 cats。

`cat|dog` 按书写顺序尝试两个分支，匹配到 cat 和 dog。默认区分大小写，所以 CAT 没有被匹配。

选择也是从左到右，当前位置能匹配上哪一个就用哪一个。`a|aa|aaa` 每次只吃掉一个 a，四次得到四个 `a`。`aaa|aa|a` 先尝试三个 a，成功后再对剩下的一个 a 用最后那个分支，所以结果是 `aaa` 和 `a`。

量词都作用在它前面的 `a` 上：

- `*` 是零次或多次，`ct`、`cat`、`caat`、`caaat` 都能匹配。
- `+` 是一次或多次，`ct` 里没有 a，匹配不上。
- `?` 是零次或一次，只留下 `ct` 和 `cat`。
- `{2}` 恰好两次，只有 `caat`。`caaat` 里是三个 a，对不上。
- `{1,2}` 是一次或两次，匹配 `cat` 和 `caat`。

## 实验2：分组、回溯引用与替换

In [3]:
email_text = " 联系： alice@example.com ；备选： bob@test.org"
emails = re.findall(r"([A-Za-z0-9._+-]+)@([A-Za-z0-9.-]+)", email_text)
print("邮箱原文:", email_text)
for username, domain in emails:
    print(f"用户名 {username}，域名 {domain}")

repeat_text = "cat cat, dog cat, bird bird"
print("重复词原文:", repeat_text)
for match in re.finditer(r"\b(\w+)\s+\1\b", repeat_text):
    print(f"连续重复: {match.group(0)}，单词是 {match.group(1)}")

date_text = "2024-05-01\n2024/05/02\n2024.05.03"
converted_dates = re.sub(r"(\d{4})[-/.](\d{2})[-/.](\d{2})", r"\2-\3-\1", date_text)
print("日期转换前:")
print(date_text)
print("日期转换后:")
print(converted_dates)
print("行数:", date_text.count("\n") + 1, "->", converted_dates.count("\n") + 1)

phone_text = " 联系我： 123-456-7890 或  987.654.3210"
normalized_phones = re.sub(r"(\d{3})[-.](\d{3})[-.](\d{4})", r"(\1) \2-\3", phone_text)
print("电话原文:", phone_text)
print("电话替换后:", normalized_phones)

邮箱原文:  联系： alice@example.com ；备选： bob@test.org
用户名 alice，域名 example.com
用户名 bob，域名 test.org
重复词原文: cat cat, dog cat, bird bird
连续重复: cat cat，单词是 cat
连续重复: bird bird，单词是 bird
日期转换前:
2024-05-01
2024/05/02
2024.05.03
日期转换后:
05-01-2024
05-02-2024
05-03-2024
行数: 3 -> 3
电话原文:  联系我： 123-456-7890 或  987.654.3210
电话替换后:  联系我： (123) 456-7890 或  (987) 654-3210


邮箱用两个捕获组：`@` 左边是用户名，右边是域名。`findall` 在有分组时返回各组的内容，所以得到 `(alice, example.com)` 和 `(bob, test.org)`。

`\b(\w+)\s+\1\b` 里的 `\1` 表示再出现一次刚才捕获的那个单词。`cat cat` 和 `bird bird` 是紧挨着的重复。`dog cat` 两个词不一样，不会匹配。

日期先捕获年、月、日，替换时按 `月-日-年` 重排。`-`、`/`、`.` 三种分隔符都接受，换行没有被改掉，所以仍是三行。

电话把三位、三位、四位分成三组，中间原来的 `-` 或 `.` 都换成统一写法 `(123) 456-7890`。

## 实验3：使用 BeautifulSoup 解析 HTML


In [4]:
from bs4 import BeautifulSoup

html_doc = """
<html><head><title> 实验课程 </title></head><body>
<h1> 数据分析 </h1>
<p class="course"> 第六讲：正则表达式 </p>
<p class="course"> 第七讲：结构化数据 </p>
<a href="https://example.org/lecture06"> 讲义六 </a>
<a href="https://example.org/lecture07"> 讲义七 </a>
</body></html>
"""

soup = BeautifulSoup(html_doc, "html.parser")
print("页面标题:", soup.title.get_text())
heading = soup.h1
print("h1 标签名:", heading.name)
print("h1 文本:", heading.get_text())

courses = soup.find_all("p", class_="course")
print("class=course 的段落:")
for paragraph in courses:
    print(repr(paragraph.get_text()))

links = soup.find_all("a")
print("链接:")
for link in links:
    print(repr(link.get_text()), "->", link.get("href"))

print("整个文档 get_text():", repr(soup.get_text()))
print("单个段落 get_text():", repr(courses[0].get_text()))
print("单个链接 get_text():", repr(links[0].get_text()))

页面标题:  实验课程 
h1 标签名: h1
h1 文本:  数据分析 
class=course 的段落:
' 第六讲：正则表达式 '
' 第七讲：结构化数据 '
链接:
' 讲义六 ' -> https://example.org/lecture06
' 讲义七 ' -> https://example.org/lecture07
整个文档 get_text(): '\n 实验课程 \n 数据分析 \n 第六讲：正则表达式 \n 第七讲：结构化数据 \n 讲义六 \n 讲义七 \n\n'
单个段落 get_text(): ' 第六讲：正则表达式 '
单个链接 get_text(): ' 讲义六 '


标题文本是“实验课程”。`h1` 的标签名是 `h1`，文本是“数据分析”。

`find_all("p", class_="course")` 找到两段课程介绍。`find_all("a")` 找到两个链接，可见文字和 `href` 要分开看：文字是“讲义六”“讲义七”，地址在属性里。

`get_text()` 只取标签之间能看见的文字，不包含标签名和属性。整篇文档会把标题、一级标题、两个段落和两个链接文字按顺序接在一起。单个段落只有那一讲的文字，单个链接只有链接文字，没有 `https://` 地址。